# 03 — Evaluation (local, CPU)

Reads the downloaded `02_train_<arch>_output/` folders and computes every metric from `preds.csv`.
Per run, the temperature T and the 95%-sensitivity threshold are fitted on **val** and frozen; test is only scored.

All metrics use the `logit` column. The `prob` column saturates to exactly 1.0 in float32, and the ties
lower AUROC slightly (so `test_auc` in meta.json/summary.csv is not the number to report).

Outputs go to `results/` (small CSVs, safe to commit).

In [ ]:
from glob import glob
from pathlib import Path
import numpy as np, pandas as pd
from ppneu import metrics as M

pd.set_option("display.width", 250)
REPO = Path("..") if Path("../manifests").exists() else Path(".")
ROOTS = sorted(glob(str(REPO / "02_train_*_output")))
OUT = REPO / "results"; OUT.mkdir(exist_ok=True)
N_BOOT = 2000

preds = M.load_preds(ROOTS)
manifests = {s: pd.read_csv(REPO / "manifests" / f"{s}.csv") for s in ["official", "image_random", "grouped"]}
print(len(ROOTS), "output folders |", preds.groupby(["split", "arch", "seed"]).ngroups, "runs")

## Integrity: predictions match the frozen manifests

In [ ]:
for (s, a, k), run in preds.groupby(["split", "arch", "seed"]):
    m = manifests[s].set_index("image_id")
    for fold in ("val", "test"):
        r = run[run.fold == fold]
        assert set(r.image_id) == set(m.index[m.fold == fold]), (s, a, k, fold)
        assert (r.y.values == m.loc[r.image_id, "y"].values).all(), (s, a, k, fold)
print("ok")
display(M.run_meta(ROOTS).groupby("arch")[["best_epoch", "epochs_run", "minutes"]].agg(["min", "max"]).round(1))

## Per-run metrics (T and threshold fitted on val)

In [ ]:
runs = M.evaluate_all(preds, target_sens=0.95)
runs.to_csv(OUT / "run_metrics.csv", index=False)
summary = M.summarize(runs)
summary.round(4).to_csv(OUT / "summary_by_split_arch.csv")
display(summary.round(4))

## Table 2a — leakage: image_random vs grouped (different test sets)
Mean over seeds; `all_seeds_higher` = every image_random seed beats every grouped seed.

In [ ]:
a = runs.pivot_table(index="arch", columns="split", values="auroc")
lo = runs.groupby(["arch", "split"]).auroc.min().unstack()
hi = runs.groupby(["arch", "split"]).auroc.max().unstack()
t2a = pd.DataFrame({
    "auroc_image_random": a.image_random, "auroc_grouped": a.grouped,
    "error_ratio": (1 - a.grouped) / (1 - a.image_random),
    "all_seeds_higher": lo.image_random > hi.grouped,
    "sens_image_random": runs[runs.split == "image_random"].groupby("arch").sens.mean(),
    "sens_grouped": runs[runs.split == "grouped"].groupby("arch").sens.mean(),
})
t2a.round(4).to_csv(OUT / "table2a_leakage_splits.csv")
display(t2a.round(4))

## Table 2b — leakage within one test set (image_random)
Test images whose patient cluster is also in train ("seen") vs not. Prevalence differs a lot between the
two groups, so errors are compared **per class**. Each image's error rate is averaged over all runs
(5 archs × 3 seeds, decision at the val-fitted 95%-sensitivity threshold), then images are bootstrapped.

In [ ]:
seen = M.seen_in_train(manifests["image_random"])
thr = runs.set_index(["split", "arch", "seed"]).threshold_logit
t = preds[(preds.split == "image_random") & (preds.fold == "test")].copy()
t["thr"] = [thr[("image_random", a, k)] for a, k in zip(t.arch, t.seed)]
t["err"] = ((t.logit >= t.thr) != (t.y == 1)).astype(float)
t["err_05"] = ((t.logit >= 0) != (t.y == 1)).astype(float)
t["seen"] = seen.reindex(t.image_id).to_numpy(dtype=bool)
img = t.groupby("image_id").agg(y=("y", "first"), seen=("seen", "first"),
                                err=("err", "mean"), err_05=("err_05", "mean"))

mean_err = lambda y, e: float(np.mean(e))
rows = []
for cls, name in [(1, "pneumonia"), (0, "normal")]:
    for col, rule in [("err", "val 95% sens"), ("err_05", "0.5")]:
        s_, u_ = img[(img.y == cls) & img.seen], img[(img.y == cls) & ~img.seen]
        d, l, h, p = M.bootstrap_diff(mean_err, u_.y, u_[col], s_.y, s_[col], n_boot=N_BOOT)
        rows.append({"class": name, "rule": rule, "n_unseen": len(u_), "n_seen": len(s_),
                     "err_unseen_%": 100 * u_[col].mean(), "err_seen_%": 100 * s_[col].mean(),
                     "diff_pts": 100 * d, "ci_lo": 100 * l, "ci_hi": 100 * h, "p": p})
t2b = pd.DataFrame(rows)
t2b.round(3).to_csv(OUT / "table2b_leakage_seen_unseen.csv", index=False)
display(t2b.round(3))

per_arch = (t.assign(cls=t.y.map({1: "pneumonia", 0: "normal"}))
              .groupby(["arch", "cls", "seen"]).err.mean().mul(100).unstack(["cls", "seen"]).round(2))
display(per_arch)

## Calibration
T > 1 means overconfident. Temperature scaling changes probabilities only, so it is judged by log-loss and Brier;
binned ECE barely moves because most predictions sit near 0 or 1.

In [ ]:
cal = runs.groupby("split")[["T", "nll", "nll_ts", "brier", "brier_ts", "ece", "ece_ts"]].agg(["mean", "std"])
cal.round(4).to_csv(OUT / "calibration_by_split.csv")
display(cal.round(4))

## Official test set (distribution shift)
Same models, two operating points. The val-fitted thresholds come from a clean, grouped val fold.

In [ ]:
off = runs[runs.split == "official"].groupby("arch")[
    ["auroc", "sens", "spec", "sens_05", "spec_05", "nll", "ece", "threshold_prob"]].agg(["mean", "std"])
off.round(4).to_csv(OUT / "official_shift.csv")
display(off.round(4))

## Per-run AUROC with bootstrap 95% CI (seed 0)

In [ ]:
rows = []
for (s, a, k), run in preds[(preds.fold == "test") & (preds.seed == 0)].groupby(["split", "arch", "seed"]):
    est, lo, hi = M.bootstrap_ci(M.auroc, run.y, run.logit, n_boot=N_BOOT)
    rows.append({"split": s, "arch": a, "auroc": est, "ci_lo": lo, "ci_hi": hi})
ci = pd.DataFrame(rows)
ci.round(4).to_csv(OUT / "auroc_ci_seed0.csv", index=False)
display(ci.pivot_table(index="arch", columns="split", values=["auroc", "ci_lo", "ci_hi"]).round(4))